In [1]:
import pathlib
import textwrap

import numpy as np
import pickle
from PIL import Image

from IPython.display import display
from IPython.display import Markdown

# from matplotlib.pyplot import imshow

# from sklearn.metrics import roc_auc_score
import re
import PIL.Image
import json
import os
import pandas as pd


def to_markdown(text):
  text = text.replace('•', '  *')
  return Markdown(textwrap.indent(text, '> ', predicate=lambda _: True))

from google import genai

In [2]:
all_files = []
img_dir = """/projects/matsci/vlm_microscopy/Microscopy/NFFA/sampled_counting_cropped"""
for path, subdirs, files in os.walk(img_dir):
    for name in files:
        all_files.append(os.path.join(path, name))

In [ ]:
client = genai.Client(api_key='XYZ')

In [4]:
print("List of models that support generateContent:\n")
for m in client.models.list():
    for action in m.supported_actions:
        if action == "generateContent":
            print(m.name)

List of models that support generateContent:

models/gemini-2.5-pro-preview-03-25
models/gemini-2.5-flash-preview-05-20
models/gemini-2.5-flash
models/gemini-2.5-flash-lite-preview-06-17
models/gemini-2.5-pro-preview-05-06
models/gemini-2.5-pro-preview-06-05
models/gemini-2.5-pro
models/gemini-2.0-flash-exp
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-exp-image-generation
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.0-flash-preview-image-generation
models/gemini-2.0-flash-lite-preview-02-05
models/gemini-2.0-flash-lite-preview
models/gemini-2.0-pro-exp
models/gemini-2.0-pro-exp-02-05
models/gemini-exp-1206
models/gemini-2.0-flash-thinking-exp-01-21
models/gemini-2.0-flash-thinking-exp
models/gemini-2.0-flash-thinking-exp-1219
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/learnlm-2.0-flash-experimental
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models

In [5]:
safety_settings = [{"category": "HARM_CATEGORY_SEXUALLY_EXPLICIT", "threshold": "BLOCK_NONE"},
                   {"category": "HARM_CATEGORY_HATE_SPEECH", "threshold": "BLOCK_NONE"}, 
                   {"category": "HARM_CATEGORY_HARASSMENT", "threshold": "BLOCK_NONE"},
                   {"category": "HARM_CATEGORY_DANGEROUS_CONTENT", "threshold": "BLOCK_NONE"}]

generation_settings = {"top_p": 0.7, "max_output_tokens": 1024}

In [6]:
all_files[-1]

'/projects/matsci/vlm_microscopy/Microscopy/NFFA/sampled_counting_cropped/Fibres/L9_c14d469086dc04d60a9ed1ffa00def97.jpg'

In [7]:
last_processed_idx = -1
unprocessed_ids = []
responses = []

In [8]:
import time

In [9]:
for idx, file in enumerate(all_files):
    if idx <= last_processed_idx:
        continue
    img = PIL.Image.open(file)
    print(f"Processing image: {file}")
    prompt = """This is an SEM image of fibres or particles. Please count the number of fibres or particles in this image in the format 'Count: <count>'. If you are unable to count, please write NaN."""
    response = client.models.generate_content(model = 'gemini-2.5-pro', contents = [img, prompt])
    try:
        answer = response.text
    except Exception as e:
        unprocessed_ids.append(path)
        answer = "MODEL ERROR"
    print(answer)
    print()
    responses.append(answer)
    last_processed_idx = idx
    time.sleep(60)

Processing image: /projects/matsci/vlm_microscopy/Microscopy/NFFA/sampled_counting_cropped/Particles/L2_0a9d325c8250edd1e89670d17b0b04ec.jpg
NaN

Processing image: /projects/matsci/vlm_microscopy/Microscopy/NFFA/sampled_counting_cropped/Particles/0f2e62666816023d0cf091eb81f552d1.jpg
Count: 28

Processing image: /projects/matsci/vlm_microscopy/Microscopy/NFFA/sampled_counting_cropped/Particles/L2_0f1f7aa38aa58827081233f05aba98b0.jpg
Count: 19

Processing image: /projects/matsci/vlm_microscopy/Microscopy/NFFA/sampled_counting_cropped/Particles/L2_0c02a5fbd18b5cefc1115295ba1d4fc4.jpg
Count: 7

Processing image: /projects/matsci/vlm_microscopy/Microscopy/NFFA/sampled_counting_cropped/Particles/L2_0d199c2f1c05716d2004b04c844504bc.jpg
Count: 11

Processing image: /projects/matsci/vlm_microscopy/Microscopy/NFFA/sampled_counting_cropped/Particles/L2_0a7e0f7bb2040886461cf4cb65318849.jpg
Count: 7

Processing image: /projects/matsci/vlm_microscopy/Microscopy/NFFA/sampled_counting_cropped/Particle

In [11]:
# f8365919f611dcc61903c5ef7093ba6f

In [12]:
len(responses)

51

In [13]:
response.candidates

[Candidate(
   content=Content(
     parts=[
       Part(
         text='Count: 4'
       ),
     ],
     role='model'
   ),
   finish_reason=<FinishReason.STOP: 'STOP'>,
   index=0
 )]

In [14]:
response.prompt_feedback

In [15]:
len(unprocessed_ids)

0

In [16]:
with open('results_nffa_sampled_data_counting.pkl', 'wb') as f:
    pickle.dump({"responses": responses, "unprocessed_ids": unprocessed_ids}, f)

In [17]:
with open('results_nffa_sampled_data_counting.pkl', 'rb') as f:
    data = pickle.load(f)
    responses = data['responses']

In [18]:
counts = [(elem, sum(np.array(responses) == elem)) for elem in np.unique(np.array(responses))]

In [19]:
counts

[(np.str_('Based on the provided SEM image, it is not possible to provide an accurate count of all "fibres or particles".\n\nHere is a breakdown of the components in the image:\n1.  **Fibres:** There are approximately 4 large, distinct fibres visible in the image frame.\n2.  **Particles:** The image contains a vast number of particles of varying sizes. There are larger, crystalline-like growths agglomerated on the fibres, and the entire background is covered with a dense field of extremely fine particles that are too numerous to count individually.\n\nBecause the particles are innumerable, a total count of "fibres or particles" cannot be determined.\n\nCount: NaN'),
  np.int64(1)),
 (np.str_('Count: 1'), np.int64(5)),
 (np.str_('Count: 11'), np.int64(2)),
 (np.str_('Count: 12'), np.int64(2)),
 (np.str_('Count: 13'), np.int64(4)),
 (np.str_('Count: 14'), np.int64(1)),
 (np.str_('Count: 16'), np.int64(1)),
 (np.str_('Count: 17'), np.int64(1)),
 (np.str_('Count: 19'), np.int64(1)),
 (np.s

In [20]:
len(unprocessed_ids)

0

In [21]:
unprocessed_ids

[]

In [23]:
actuals = []
predictions = []

# for file in all_files:
#     actuals.append(int(file.split("_")[-4][1:]))
    
import re

predictions = []
for idx, path in enumerate(all_files):
    numbers = re.findall(r'\d+', responses[idx])
    pred = -1
    if len(numbers) > 0:
        pred = int(numbers[0])
    predictions.append(pred)

In [24]:
len(actuals)

0

In [25]:
len(predictions)

51

In [26]:
import pandas as pd
df = pd.DataFrame({'img_path': all_files, 'raw_predictions': responses,  'predictions': predictions})

In [27]:
df.to_csv('counting_NFFA_gemini_manually_sampled.csv')